# Splitting the all-sky survey

Here's the idea: we'd have telescope at each site doing a high-cadence equitorial survey, and the other doing the "filler" survey to fill in the gaps.

I want to do some simulations to see how that would look, using the FAST/HEATS surveys as a starting point.

In [ ]:
from gtecs.obs import database as db
from gototile.grid import SkyGrid

## Surveys

Let's start by getting which tiles have been in the FAST and HEATS surveys.

In [ ]:
with db.session_manager() as s:
    svs = s.query(db.Survey).filter(db.Survey.name.like('%survey%')).all()
    for sv in svs:
        print(sv.name, len(sv.targets))

In [ ]:
survey_tiles = {}
with db.session_manager() as s:
    svs = s.query(db.Survey).filter(db.Survey.name.like('%INT%')).all()
    for sv in svs:
        print(sv.name, len(sv.targets))
        survey_tiles[sv.name] = []
        for t in sv.targets:
            survey_tiles[sv.name].append(t.grid_tile.name)

    svs = s.query(db.Survey).filter(db.Survey.name.like('%FAST%')).all()
    for sv in svs:
        print(sv.name, len(sv.targets))
        survey_tiles[sv.name] = []
        for t in sv.targets:
            survey_tiles[sv.name].append(t.grid_tile.name)

    svs = s.query(db.Survey).filter(db.Survey.name.like('%fast%')).all()
    for sv in svs:
        print(sv.name, len(sv.targets))
        survey_tiles[sv.name] = []
        for t in sv.targets:
            survey_tiles[sv.name].append(t.grid_tile.name)

    sv = s.query(db.Survey).filter(db.Survey.name=='goto-heats_survey').one()
    print(sv.name, len(sv.targets))
    survey_tiles[sv.name] = []
    for t in sv.targets:
        survey_tiles[sv.name].append(t.grid_tile.name)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

grid: SkyGrid = SkyGrid.from_name('GOTO')
fig = plt.figure(figsize=(8, 10))

# -----------------------
for i, survey in enumerate(survey_tiles):
    print(survey, len(survey_tiles[survey]))

    axes = fig.add_subplot(100*len(survey_tiles) + 11+i, projection='astro hours mollweide')
    axes.grid()

    grid.plot_tiles(axes, tilenames=set(survey_tiles[survey]), fc='tab:blue',
                            ec='none', alpha=0.7, zorder=1)
    grid.plot_tiles(axes, fc='none', ec='0.3', lw=0.3, zorder=1.5)

    axes.set_title(survey)

# -----------------------
fig.tight_layout()
plt.show()

HEATS is the best place to start, since it has been extended to all RAs.

So we basically want to consider HEATS and an extra Anti-HEATS survey, that has all the tiles not in HEATS.

**HEATS** stands for **High-cadence Equatorial Astronomical Transient Survey**.

So I'm going to call the anti-HEATS survey "**COLD**": **Complementary Observations for Low-cadence Detections**.

I guess it should be _COLDS_ to match HEATS, but that reminds me too much of being ill. :(

In [ ]:
heats_tiles = set(survey_tiles['goto-heats_survey'])
cold_tiles = set(grid.tilenames) - heats_tiles
cold_tiles -= set(grid.tilenames[:5])  # Remove the south pole
cold_tiles -= set(grid.tilenames[-5:])  # Remove the north pole

print(len(heats_tiles), len(heats_tiles)/len(grid.tilenames))
print(len(cold_tiles), len(cold_tiles)/len(grid.tilenames))

Right, so near enough 30% of the sky is in HEATS, so the other 70% would be in COLD.

## Simulations

I have plenty of notebooks with survey simulation code, so let's find some of that.
I can adapt the gototile cli script...


In [ ]:
import numpy as np
from astropy import units as u
from astropy.coordinates import AltAz, EarthLocation, SkyCoord, get_sun
from astropy.time import Time, TimeDelta

def simulate_survey(
        start_date, end_date, tel_tile_masks,
        pointing_time=5*60, twilight=-12, min_alt=30,
    ):
    """Simulate a survey over the given time period."""
    print(f'Simulating from {start_date} to {end_date}...')

    # Create sites and grid
    sites = {
        'N': EarthLocation.of_site('lapalma'),
        'S': EarthLocation.of_site('sso'),
    }
    grid = SkyGrid.from_name('GOTO')

    # Run loop over time
    date = start_date
    observations = {t: [] for t in grid.tilenames}
    while date < end_date:
        print(date.iso, end=' - ')

        for site in sites:
            print(f'{site}', end=' ')
            # First check if the Sun is above the horizon
            altaz_frame = AltAz(obstime=date, location=sites[site])
            sun = get_sun(date)
            if sun.transform_to(altaz_frame).alt > twilight * u.deg:
                print('daytime', end='                       ')
                continue

            # Go through each telescope and pick what it should observe
            for tel in [1, 2]:
                tel = tel if site == 'N' else tel + 2
                print(f'{tel}:', end=' ')

                # Get the mask for the tiles this telescope can observe
                mask = tel_tile_masks[tel]

                # First, get the altitude of all the tiles at this time
                coords = grid.coords[mask]
                altitudes = coords.transform_to(altaz_frame).alt

                # Find which are above the horizon
                visible_mask = altitudes > min_alt * u.deg
                if sum(visible_mask) == 0:
                    if verbose:
                        print('NULL', end='        ')
                    continue

                # Get the count of how many times each visible tile has been observed
                obs_count = np.array([
                    len(observations[t])
                    for t in np.array(grid.tilenames)[mask][visible_mask]
                ])

                # Select the tiles that have been observed the least,
                # and use the altitude (airmass) as a tiebreaker
                selected_mask = obs_count == min(obs_count)
                selected_altitudes = altitudes[visible_mask][selected_mask]
                best_selected_index = selected_altitudes.argmin()
                best_tile = np.array(grid.tilenames)[mask][visible_mask][selected_mask][best_selected_index]
                best_tile_index = np.where(np.array(grid.tilenames) == best_tile)[0][0]

                print(f'{best_tile} (n={len(observations[best_tile])})', end=' ')

                # Mark that tile as observed at this time by this telescope
                observations[best_tile].append((date, tel))

        date += TimeDelta(pointing_time * u.s)
        print()

    return observations

In [ ]:
start_date = Time('2025-11-18 16:00')  # End of O4
end_date = Time('2026-05-18 16:00')

# Define which tiles each telescope can observe
heats_mask = np.array([tile in heats_tiles for tile in grid.tilenames])
cold_mask = np.array([tile in cold_tiles for tile in grid.tilenames])
tel_tile_masks = {
    1: heats_mask,
    2: cold_mask,
    3: heats_mask,
    4: cold_mask,
}

# Run the simulation
observations = simulate_survey(start_date, end_date, tel_tile_masks)

### Plots

Okay, we have some results. Let's plot them.


In [ ]:
from matplotlib.gridspec import GridSpec

def plot_survey_results(observations):
    """Plot the results of the survey simulation."""
    obs_count = np.array([len(observations[t]) for t in observations])

    obs_cadence = np.array([
        [(observations[t][i][0] - observations[t][i-1][0]).to(u.day).value
        for i in range(len(observations[t]))
        if i > 0]
        for t in observations
    ], dtype=object)

    median_cadence = np.array([
        np.median(obs_cadence[i]) if len(obs_cadence[i]) > 1 else 0
        for i in range(len(obs_cadence))
    ])

    # -------------------------
    fig = plt.figure(figsize=(12, 8), dpi=200)
    fig.suptitle(f'Start time: {start_date.iso}\nEnd time: {end_date.iso}')

    gs = GridSpec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])

    # -------------------------
    axes = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
    axes.grid()

    tiles = grid.plot_tiles(axes, array=obs_count, cmap='viridis',
                            ec='none', alpha=0.9, zorder=0.6)
    if len(obs_count[obs_count == 0]) > 0:
        grid.plot_tiles(axes, tilenames=np.array(grid.tilenames)[np.array(obs_count) == 0],
                        fc='0.4', ec='none', zorder=0.7)

    grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
    grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)

    count_ticks = np.arange(0, max(obs_count) + 1, 10 if max(obs_count) > 100 else 2)
    fig.colorbar(
        tiles, ax=axes, ticks=count_ticks,
        fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
        label=f'Total number of observations'
        f'\n(min={min(obs_count):.0f}, '
        f'mean={np.mean(obs_count):.1f}, '
        f'max={max(obs_count):.0f})'
    )

    # legend_elements = [
    #     Patch(facecolor='none', edgecolor='red', label='HEATS tiles'),
    #     Patch(facecolor='none', edgecolor='blue', label='COLD tiles'),
    # ]
    # axes.legend(handles=legend_elements, loc='lower center',
    #             bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

    # -------------------------
    axes = fig.add_subplot(gs[1, 0], projection='astro hours mollweide')
    axes.grid()

    tiles = grid.plot_tiles(axes, array=median_cadence*24, cmap='viridis',
                            ec='none', alpha=0.9, zorder=0.6)
    grid.plot_tiles(axes, tilenames=np.array(grid.tilenames)[np.array(median_cadence) == 0],
                    fc='0.4', ec='none', zorder=0.7)

    grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
    grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)

    cadence_ticks = np.arange(0, int(max(median_cadence*24))+1, 12)
    fig.colorbar(
        tiles, ax=axes, ticks=cadence_ticks,
        fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
        label='Median hours between observations'
        f'\n(min={min(median_cadence[median_cadence > 0]*24):.1f}, '
        f'mean={np.mean(median_cadence[median_cadence > 0]*24):.1f}, '
        f'max={max(median_cadence*24):.1f})'
    )

    # legend_elements = [
    #     Patch(facecolor='none', edgecolor='red', label='HEATS tiles'),
    #     Patch(facecolor='none', edgecolor='blue', label='COLD tiles'),
    # ]
    # axes.legend(handles=legend_elements, loc='lower center',
    #             bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

    # -------------------------
    axes = fig.add_subplot(gs[0, 1])
    counts, bins, patches = axes.hist([obs_count[heats_mask], obs_count[~heats_mask]],
            bins=np.arange(0, int(max(obs_count))+1), stacked=True, width=0.9,
            color=['red', 'blue'],
            label=['Tiles in the HEATS survey (n={})'.format(len(heats_tiles)),
                   'Tiles in the COLD survey (n={})'.format(len(cold_tiles))],
    )
    axes.legend(loc='upper right', bbox_to_anchor=(1, 1.25), frameon=False)
    axes.set_xlabel('Number of observations')
    axes.set_ylabel('Frequency')
    axes.set_xlim(0-0.5, int(max(obs_count))+1+0.5)
    axes.set_xticks(count_ticks)

    # -------------------------
    axes = fig.add_subplot(gs[1, 1])
    #axes.hist([median_cadence[heats_mask]*24, median_cadence[~heats_mask]*24],
    axes.hist([[t*24 for cadences in obs_cadence[heats_mask] for t in cadences],
            [t*24 for cadences in obs_cadence[~heats_mask] for t in cadences]],
            bins=np.arange(0, int(max(median_cadence*24))+1), stacked=True, width=0.95,
            color=['red', 'blue'],
            #label=['Tiles in the HEATS survey', 'Tiles in the COLD survey'],
    )
    #axes.legend()
    axes.set_xlabel('Hours between observations')
    axes.set_ylabel('Frequency')
    axes.set_xlim(0-0.1, int(max(median_cadence*24))+1+0.1)
    axes.set_xticks(cadence_ticks)

    # -------------------------
    #fig.tight_layout()
    plt.show()

In [ ]:
plot_survey_results(observations)

## What about the all-sky survey?

Let's do a comparison.

In [ ]:
allsky_tiles = set(grid.tilenames)
allsky_tiles -= set(grid.tilenames[:5])  # Remove the south pole
allsky_tiles -= set(grid.tilenames[-5:])  # Remove the north pole
allsky_mask = np.array([tile in allsky_tiles for tile in grid.tilenames])

tel_tile_masks = {
    1: allsky_mask,
    2: allsky_mask,
    3: allsky_mask,
    4: allsky_mask,
}

observations = simulate_survey(start_date, end_date, tel_tile_masks)

In [ ]:
plot_survey_results(observations)

TODO:
* Remove X% of time for transients, other surveys etc.
* Use real times from the ObsDB, e.g. including weather downtime.
* Try different "HEATS" surveys with different dec cuts.
